In [1]:
import gzip
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math
def compute_ln(x):
    """Natural logarithm with zero handling"""
    return -math.inf if x == 0 else math.log(x)
class HMMAnalyzer:
    def __init__(self):
        self.states = ['E', '5', 'I']
        self.transitions = {
            'Start': {'E': 1.0},
            'E': {'E': 0.9, '5': 0.1},
            '5': {'I': 1.0},
            'I': {'I': 0.9, 'End': 0.1}
        }
        self.emissions = {
            'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
            '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
            'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
        }
    def evaluate_sequence_probability(self, state_chain, dna_string):
        """Calculate log probability of a state path generating a DNA sequence"""
        if len(state_chain) != len(dna_string):
            raise ValueError("State chain and DNA string length mismatch")
        cumulative_logp = 0.0
        prev = 'Start'
        for idx in range(len(dna_string)):
            current_state = state_chain[idx]
            base = dna_string[idx]
            tr_prob = self.transitions[prev].get(current_state, 0)
            em_prob = self.emissions[current_state].get(base, 0)
            cumulative_logp += compute_ln(tr_prob) + compute_ln(em_prob) 
            prev = current_state
        if prev == 'I':
            cumulative_logp += compute_ln(self.transitions['I']['End'])
        return round(cumulative_logp, 2)
    def find_best_path(self, dna_sequence):
        """Dynamic programming implementation of Viterbi algorithm"""
        seq_len = len(dna_sequence)
        dp_table = [{} for _ in range(seq_len)]
        backpointer = {}
        for s in ['E']:
            dp_table[0][s] = (
                compute_ln(self.transitions['Start'][s]) + 
                compute_ln(self.emissions[s][dna_sequence[0]]))
            backpointer[s] = [s]
        for t in range(1, seq_len):
            current_paths = {}
            for curr_s in self.states:
                max_p = -math.inf
                best_prev_s = None
                for prev_s in dp_table[t-1]:
                    if curr_s in self.transitions.get(prev_s, {}):
                        trans_p = compute_ln(self.transitions[prev_s][curr_s])
                        emit_p = compute_ln(self.emissions[curr_s][dna_sequence[t]])
                        total_p = dp_table[t-1][prev_s] + trans_p + emit_p
                        if total_p > max_p:
                            max_p = total_p
                            best_prev_s = prev_s
                if best_prev_s is not None:
                    dp_table[t][curr_s] = max_p
                    current_paths[curr_s] = backpointer[best_prev_s] + [curr_s]
            backpointer = current_paths
        final_state = max(dp_table[-1], key=dp_table[-1].get)
        return ''.join(backpointer[final_state]), round(dp_table[-1][final_state], 2)
if __name__ == "__main__":
    analyzer = HMMAnalyzer()
    test_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
    test_seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
    
    print(f"Path probability: {analyzer.evaluate_sequence_probability(test_path, test_seq)}")
    optimal_path, path_prob = analyzer.find_best_path(test_seq)
    print(f"Optimal path: {optimal_path}")
    print(f"Path probability: {path_prob}")

Path probability: -41.22
Optimal path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Path probability: -38.68
